# Comment Category Prediction — v5 ULTRA MEMORY SAFE
- Char TF-IDF completely removed
- Word TF-IDF: 15k features only
- Aggressive gc.collect() everywhere
- Float32 throughout

In [ ]:
import numpy as np
import pandas as pd
import re, gc, psutil, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def ram():
    v = psutil.virtual_memory()
    print(f'RAM: {v.used/1e9:.1f} GB used / {v.total/1e9:.1f} GB total ({v.percent:.0f}%)')

print('Libraries ready!')
ram()

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/test.csv')
print('Train:', train.shape, '| Test:', test.shape)
print(train['label'].value_counts())
ram()

In [ ]:
CONTRACTIONS = {
    "won't":"will not","can't":"cannot","n't":" not",
    "'re":" are","'ve":" have","'ll":" will","'d":" would","'m":" am",
    " u ":" you "," ur ":" your "," r ":" are ",
    " omg ":" oh my god "," lol ":" laugh "," idk ":" i do not know ",
    " wtf ":" what the f "," stfu ":" shut up "," tbh ":" to be honest ",
}

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = text.encode('ascii','ignore').decode('ascii').lower()
    for k,v in CONTRACTIONS.items(): text = text.replace(k,v)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

print('clean_text ready!')

In [ ]:
VIOLENT_KW   = ['kill','die','dead','murder','shoot','attack','hurt','harm','violence','stab','torture']
OFFENSIVE_KW = ['stupid','idiot','moron','dumb','loser','pathetic','trash','worthless','hate','useless']
IDENTITY_KW  = ['race','black','white','muslim','jew','gay','lesbian','trans','gender','religion']

# Post stats — train only
post_stats = train.groupby('post_id')['label'].agg(
    post_label_mode   = lambda x: x.mode()[0],
    post_label_mean   = 'mean',
    post_label_std    = 'std',
    post_label_count  = 'count',
    post_label_0_frac = lambda x: (x==0).mean(),
    post_label_1_frac = lambda x: (x==1).mean(),
    post_label_2_frac = lambda x: (x==2).mean(),
    post_label_3_frac = lambda x: (x==3).mean(),
).reset_index()
post_stats['post_label_std'] = post_stats['post_label_std'].fillna(0)

FALLBACK = {
    'post_label_mode':   int(train['label'].mode()[0]),
    'post_label_mean':   train['label'].mean(),
    'post_label_std':    train['label'].std(),
    'post_label_count':  1,
    'post_label_0_frac': train['label'].eq(0).mean(),
    'post_label_1_frac': train['label'].eq(1).mean(),
    'post_label_2_frac': train['label'].eq(2).mean(),
    'post_label_3_frac': train['label'].eq(3).mean(),
}
print('Post stats done!')

In [ ]:
def engineer(df):
    df = df.copy()

    df['created_date'] = pd.to_datetime(df['created_date'], utc=True)
    df['hour']        = df['created_date'].dt.hour.astype(np.int8)
    df['day_of_week'] = df['created_date'].dt.dayofweek.astype(np.int8)
    df['is_weekend']  = df['day_of_week'].isin([5,6]).astype(np.int8)
    df['is_night']    = ((df['hour']>=22)|(df['hour']<=5)).astype(np.int8)

    df['comment']       = df['comment'].fillna('')
    df['comment_clean'] = df['comment'].apply(clean_text)
    raw = df['comment']

    df['comment_len']   = raw.apply(len).astype(np.int32)
    df['word_count']    = raw.apply(lambda x: len(x.split())).astype(np.int16)
    df['caps_ratio']    = raw.apply(lambda x: sum(c.isupper() for c in x)/(len(x)+1)).astype(np.float32)
    df['exclamations']  = raw.apply(lambda x: x.count('!')).astype(np.int16)
    df['questions']     = raw.apply(lambda x: x.count('?')).astype(np.int16)
    df['lex_diversity'] = raw.apply(lambda x: len(set(x.lower().split()))/(len(x.split())+1)).astype(np.float32)
    df['all_caps_words']= raw.apply(lambda x: sum(1 for w in x.split() if w.isupper() and len(w)>1)).astype(np.int16)

    txt = raw.str.lower()
    df['kw_violent']   = txt.apply(lambda x: sum(w in x for w in VIOLENT_KW)).astype(np.int8)
    df['kw_offensive'] = txt.apply(lambda x: sum(w in x for w in OFFENSIVE_KW)).astype(np.int8)
    df['kw_identity']  = txt.apply(lambda x: sum(w in x for w in IDENTITY_KW)).astype(np.int8)
    df['has_violent']  = (df['kw_violent']>0).astype(np.int8)
    df['has_offensive']= (df['kw_offensive']>0).astype(np.int8)
    df['has_identity'] = (df['kw_identity']>0).astype(np.int8)

    df['log_upvote']   = np.log1p(df['upvote']).astype(np.float32)
    df['log_downvote'] = np.log1p(df['downvote']).astype(np.float32)
    df['vote_ratio']   = (df['upvote']/(df['upvote']+df['downvote']+1)).astype(np.float32)
    df['controversy']  = (df['downvote']/(df['upvote']+df['downvote']+1)).astype(np.float32)

    id_cols = ['race','religion','gender','disability']
    df[id_cols] = df[id_cols].fillna('none')
    df['identity_count'] = (df[id_cols]!='none').sum(axis=1).astype(np.int8)

    df = df.merge(post_stats, on='post_id', how='left')
    df.fillna(FALLBACK, inplace=True)
    return df

print('Engineering train...')
train = engineer(train)
print('Engineering test...')
test  = engineer(test)
gc.collect()
print('Done! Shape:', train.shape)
ram()

In [ ]:
# ONLY word TF-IDF — char TF-IDF hataaya (OOM fix)
print('Fitting TF-IDF (word only, 15k features)...')
word_tfidf = TfidfVectorizer(
    ngram_range  = (1, 2),
    max_features = 15000,
    sublinear_tf = True,
    min_df       = 3,
    strip_accents= 'unicode',
    dtype        = np.float32
)
X_text_tr = word_tfidf.fit_transform(train['comment_clean'])
X_text_te = word_tfidf.transform(test['comment_clean'])
print(f'TF-IDF shape: {X_text_tr.shape} | {X_text_tr.data.nbytes/1e6:.0f} MB')
ram()

In [ ]:
cat_cols = ['race','religion','gender','disability']
num_cols = [
    'emoticon_1','emoticon_2','emoticon_3','upvote','downvote','if_1','if_2',
    'hour','day_of_week','is_weekend','is_night',
    'comment_len','word_count','caps_ratio','exclamations','questions',
    'lex_diversity','all_caps_words',
    'kw_violent','kw_offensive','kw_identity',
    'has_violent','has_offensive','has_identity',
    'log_upvote','log_downvote','vote_ratio','controversy',
    'identity_count',
    'post_label_mode','post_label_mean','post_label_std','post_label_count',
    'post_label_0_frac','post_label_1_frac','post_label_2_frac','post_label_3_frac',
]
num_cols = [c for c in num_cols if c in train.columns]
print(f'Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}')

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_cat_tr = csr_matrix(oe.fit_transform(train[cat_cols]).astype(np.float32))
X_cat_te = csr_matrix(oe.transform(test[cat_cols]).astype(np.float32))
X_num_tr = csr_matrix(train[num_cols].values.astype(np.float32))
X_num_te = csr_matrix(test[num_cols].values.astype(np.float32))

X_full_tr = hstack([X_text_tr, X_cat_tr, X_num_tr], format='csr')
X_full_te = hstack([X_text_te, X_cat_te, X_num_te], format='csr')
y_all = train['label'].values

# Free immediately
del X_text_tr, X_text_te, X_cat_tr, X_cat_te, X_num_tr, X_num_te
gc.collect()

print(f'Train: {X_full_tr.shape} | {X_full_tr.data.nbytes/1e6:.0f} MB')
print(f'Test : {X_full_te.shape}')
ram()

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(
    X_full_tr, y_all, test_size=0.10, random_state=42, stratify=y_all
)
print('Train:', X_tr.shape, '| Val:', X_va.shape)

In [ ]:
def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 50, 250),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'subsample'        : trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.2, 0.5),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 0.01, 2.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.01, 2.0, log=True),
        'class_weight'     : 'balanced',
        'n_jobs'           : -1,
        'random_state'     : 42,
        'verbose'          : -1,
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    score = f1_score(y_va, m.predict(X_va), average='weighted')
    del m; gc.collect()
    return score

def optuna_callback(study, trial):
    print(f'  Trial {trial.number+1:02d}/15 | F1: {trial.value:.4f} | Best: {study.best_value:.4f}')

print('Starting Optuna (15 trials)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15, show_progress_bar=False, callbacks=[optuna_callback])

best_params = study.best_params
print(f'\nBest val F1 : {study.best_value:.4f}')
print(f'Best params : {best_params}')
ram()

In [ ]:
print('Val evaluation...')
val_model = lgb.LGBMClassifier(
    **best_params, class_weight='balanced',
    n_jobs=-1, random_state=42, verbose=-1
)
val_model.fit(X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

val_pred  = val_model.predict(X_va)
best_iter = val_model.best_iteration_

print('='*50)
print(f'Weighted F1 : {f1_score(y_va, val_pred, average="weighted"):.4f}')
print(f'Accuracy    : {(val_pred==y_va).mean():.4f}')
print('='*50)
print(classification_report(y_va, val_pred,
      target_names=['Normal','Identity Hate','Offensive','Violent']))
print(f'Best iteration: {best_iter}')

del val_model; gc.collect()
ram()

In [ ]:
print('Retraining on full data...')

final_params = {k:v for k,v in best_params.items() if k != 'n_estimators'}
final_params['n_estimators'] = best_iter + 50

final_model = lgb.LGBMClassifier(
    **final_params, class_weight='balanced',
    n_jobs=-1, random_state=42, verbose=-1
)
final_model.fit(X_full_tr, y_all)
print('Done!')

test_preds = final_model.predict(X_full_te)
print('Prediction distribution:')
print(pd.Series(test_preds).value_counts().sort_index())

sample_sub = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv')
sample_sub['label'] = test_preds
sample_sub.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(sample_sub.head())